In [40]:
!uv add deepagents tavily-python httpx markdownify langchain-core

10853.79s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Resolved 206 packages in 3ms
Checked 198 packages in 3ms


> ## Documentation Index
> Fetch the complete documentation index at: https://docs.langchain.com/llms.txt
> Use this file to discover all available pages before exploring further.

# Build a deep research agent

> Build a multi-step web research agent with subagent delegation

## Overview

This guide demonstrates how to build a multi-step web research agent from scratch using [Deep Agents](/oss/python/deepagents). The agent decomposes research questions into focused tasks, delegates them to specialized sub-agents, and synthesizes findings into a comprehensive report.

The agent you build will:

1. Plan research using a todo list
2. Delegate focused research tasks to sub-agents with isolated context
3. Assess search results and plan next steps as you gather information
4. Synthesize findings with proper citations into a final report

The spawned sub-agents will conduct web searches with Tavily, fetching full webpage content for analysis.

### Key concepts

This tutorial covers:

* [Subagents](/oss/python/deepagents/subagents) for parallel, context-isolated research
* Custom [tools](/oss/python/langchain/tools) for web search
* Multi-step planning with the [built-in planning tool](/oss/python/deepagents/harness#task-planning)

## Prerequisites

API keys for:

* Anthropic (Claude) or Google (Gemini)
* [Tavily](https://www.tavily.com/) for web search (optional - free tier sufficient)
* [LangSmith](https://smith.langchain.com?utm_source=docs\&utm_medium=cta\&utm_campaign=langsmith-signup\&utm_content=oss-deepagents-deep-research) for tracing (optional)

## Setup

<Steps>
  <Step title="Create project directory">
    ```bash theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    mkdir deep-research-agent
    cd deep-research-agent
    ```
  </Step>

  <Step title="Install dependencies">
    <Tabs>
      <Tab title="Claude">
        <CodeGroup>
          ```bash pip wrap theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
          pip install deepagents tavily-python httpx markdownify langchain-anthropic langchain-core
          ```

          ```bash uv wrap theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
          uv init
          uv add deepagents tavily-python httpx markdownify langchain-anthropic langchain-core
          uv sync
          ```
        </CodeGroup>
      </Tab>

      <Tab title="Gemini">
        <CodeGroup>
          ```bash pip wrap theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
          pip install deepagents tavily-python httpx markdownify langchain-google-genai langchain-core
          ```

          ```bash uv wrap theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
          uv init
          uv add deepagents tavily-python httpx markdownify langchain-google-genai langchain-core
          uv sync
          ```
        </CodeGroup>
      </Tab>
    </Tabs>
  </Step>

  <Step title="Set API keys">
    <Tabs>
      <Tab title="Claude">
        ```bash theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
        export ANTHROPIC_API_KEY="your_anthropic_api_key"
        export TAVILY_API_KEY="your_tavily_api_key"
        export LANGSMITH_API_KEY="your_langsmith_api_key"   # Optional
        ```
      </Tab>

      <Tab title="Gemini">
        ```bash theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
        export GOOGLE_API_KEY="your_google_api_key"
        export TAVILY_API_KEY="your_tavily_api_key"
        export LANGSMITH_API_KEY="your_langsmith_api_key"   # Optional
        ```
      </Tab>
    </Tabs>
  </Step>
</Steps>

## Build the agent

Create `agent.py` in your project directory:

<Steps>
  <Step title="Add tools">
    Add the custom search tool. The `tavily_search` tool uses Tavily for URL discovery, then fetches full webpage content so the agent can analyze complete sources instead of summaries.

    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    import os
    from typing import Annotated, Literal

    import httpx
    from langchain.tools import InjectedToolArg, tool
    from markdownify import markdownify
    from tavily import TavilyClient

    tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


    def fetch_webpage_content(url: str, timeout: float = 10.0) -> str:
        """Fetch webpage and convert HTML to markdown."""
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }
        try:
            response = httpx.get(url, headers=headers, timeout=timeout)
            response.raise_for_status()
            return markdownify(response.text)
        except Exception as e:
            return f"Error fetching {url}: {e!s}"


    @tool(parse_docstring=True)
    def tavily_search(
        query: str,
        max_results: Annotated[int, InjectedToolArg] = 1,
        topic: Annotated[
            Literal["general", "news", "finance"], InjectedToolArg
        ] = "general",
    ) -> str:
        """Search the web for information on a given query.

        Uses Tavily to discover relevant URLs, then fetches and returns full webpage content as markdown.

        Args:
            query: Search query to execute
            max_results: Maximum number of results to return (default: 1)
            topic: Topic filter - 'general', 'news', or 'finance' (default: 'general')

        Returns:
            Formatted search results with full webpage content
        """
        search_results = tavily_client.search(
            query,
            max_results=max_results,
            topic=topic,
        )
        result_texts = []
        for result in search_results.get("results", []):
            url = result["url"]
            title = result["title"]
            content = fetch_webpage_content(url)
            result_texts.append(f"## {title}\n**URL:** {url}\n\n{content}\n---")

        return f"Found {len(result_texts)} result(s) for '{query}':\n\n" + "\n".join(
            result_texts
        )
    ```
  </Step>

  <Step title="Add prompts">
    Add the orchestrator workflow and sub-agent prompt templates to `agent.py`:

    ```python expandable wrap theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    RESEARCH_WORKFLOW_INSTRUCTIONS = """# Research Workflow

    Follow this workflow for all research requests:

    1. **Plan**: Create a todo list with write_todos to break down the research into focused tasks
    2. **Save the request**: Use write_file() to save the user's research question to `/research_request.md`
    3. **Research**: Delegate research tasks to sub-agents using the task() tool - ALWAYS use sub-agents for research, never conduct research yourself
    4. **Synthesize**: Review all sub-agent findings and consolidate citations (each unique URL gets one number across all findings)
    5. **Write Report**: Write a comprehensive final report to `/final_report.md` (see Report Writing Guidelines below)
    6. **Verify**: Read `/research_request.md` and confirm you've addressed all aspects with proper citations and structure

    ## Research Planning Guidelines
    - Batch similar research tasks into a single TODO to minimize overhead
    - For simple fact-finding questions, use 1 sub-agent
    - For comparisons or multi-faceted topics, delegate to multiple parallel sub-agents
    - Each sub-agent should research one specific aspect and return findings

    ## Report Writing Guidelines

    When writing the final report to `/final_report.md`, follow these structure patterns:

    **For comparisons:**
    1. Introduction
    2. Overview of topic A
    3. Overview of topic B
    4. Detailed comparison
    5. Conclusion

    **For lists/rankings:**
    Simply list items with details - no introduction needed:
    1. Item 1 with explanation
    2. Item 2 with explanation
    3. Item 3 with explanation

    **For summaries/overviews:**
    1. Overview of topic
    2. Key concept 1
    3. Key concept 2
    4. Key concept 3
    5. Conclusion

    **General guidelines:**
    - Use clear section headings (## for sections, ### for subsections)
    - Write in paragraph form by default - be text-heavy, not just bullet points
    - Do NOT use self-referential language ("I found...", "I researched...")
    - Write as a professional report without meta-commentary
    - Each section should be comprehensive and detailed
    - Use bullet points only when listing is more appropriate than prose

    **Citation format:**
    - Cite sources inline using [1], [2], [3] format
    - Assign each unique URL a single citation number across ALL sub-agent findings
    - End report with ### Sources section listing each numbered source
    - Number sources sequentially without gaps (1,2,3,4...)
    - Format: [1] Source Title: URL (each on separate line for proper list rendering)
    - Example:

     Some important finding [1]. Another key insight [2].

     ### Sources
     [1] AI Research Paper: https://example.com/paper
     [2] Industry Analysis: https://example.com/analysis
    """
    ```

    ```python expandable wrap theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    RESEARCHER_INSTRUCTIONS = """You are a research assistant conducting research on the user's input topic. For context, today's date is {date}.

    Your job is to use tools to gather information about the user's input topic.
    You can use the tavily_search tool to find resources that can help answer the research question.
    You can call it in series or in parallel, your research is conducted in a tool-calling loop.

    You have access to the tavily_search tool for conducting web searches.

    Think like a human researcher with limited time. Follow these steps:

    1. **Read the question carefully** - What specific information does the user need?
    2. **Start with broader searches** - Use broad, comprehensive queries first
    3. **After each search, pause and assess** - Do I have enough to answer? What's still missing?
    4. **Execute narrower searches as you gather information** - Fill in the gaps
    5. **Stop when you can answer confidently** - Don't keep searching for perfection

    **Tool Call Budgets** (Prevent excessive searching):
    - **Simple queries**: Use 2-3 search tool calls maximum
    - **Complex queries**: Use up to 5 search tool calls maximum
    - **Always stop**: After 5 search tool calls if you cannot find the right sources

    **Stop Immediately When**:
    - You can answer the user's question comprehensively
    - You have 3+ relevant examples/sources for the question
    - Your last 2 searches returned similar information

    After each search, assess results before continuing: What key information did I find? What's missing? Do I have enough to answer? Should I search more or provide my answer?

    When providing your findings back to the orchestrator:

    1. **Structure your response**: Organize findings with clear headings and detailed explanations
    2. **Cite sources inline**: Use [1], [2], [3] format when referencing information from your searches
    3. **Include Sources section**: End with ### Sources listing each numbered source with title and URL

    Example:
    ## Key Findings

    Context engineering is a critical technique for AI agents [1]. Studies show that proper context management can improve performance by 40% [2].

    ### Sources
    [1] Context Engineering Guide: https://example.com/context-guide
    [2] AI Performance Study: https://example.com/study

    The orchestrator will consolidate citations from all sub-agents into the final report.
    """
    ```

    ```python expandable wrap theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    SUBAGENT_DELEGATION_INSTRUCTIONS = """# Sub-Agent Research Coordination

    Your role is to coordinate research by delegating tasks from your TODO list to specialized research sub-agents.

    ## Delegation Strategy

    **DEFAULT: Start with 1 sub-agent** for most queries:
    - "What is quantum computing?" -> 1 sub-agent (general overview)
    - "List the top 10 coffee shops in San Francisco" -> 1 sub-agent
    - "Summarize the history of the internet" -> 1 sub-agent
    - "Research context engineering for AI agents" -> 1 sub-agent (covers all aspects)

    **ONLY parallelize when the query EXPLICITLY requires comparison or has clearly independent aspects:**

    **Explicit comparisons** -> 1 sub-agent per element:
    - "Compare OpenAI vs Anthropic vs DeepMind AI safety approaches" -> 3 parallel sub-agents
    - "Compare Python vs JavaScript for web development" -> 2 parallel sub-agents

    **Clearly separated aspects** -> 1 sub-agent per aspect (use sparingly):
    - "Research renewable energy adoption in Europe, Asia, and North America" -> 3 parallel sub-agents (geographic separation)
    - Only use this pattern when aspects cannot be covered efficiently by a single comprehensive search

    ## Key Principles
    - **Bias towards single sub-agent**: One comprehensive research task is more token-efficient than multiple narrow ones
    - **Avoid premature decomposition**: Don't break "research X" into "research X overview", "research X techniques", "research X applications" - just use 1 sub-agent for all of X
    - **Parallelize only for clear comparisons**: Use multiple sub-agents when comparing distinct entities or geographically separated data

    ## Parallel Execution Limits
    - Use at most {max_concurrent_research_units} parallel sub-agents per iteration
    - Make multiple task() calls in a single response to enable parallel execution
    - Each sub-agent returns findings independently

    ## Research Limits
    - Stop after {max_researcher_iterations} delegation rounds if you haven't found adequate sources
    - Stop when you have sufficient information to answer comprehensively
    - Bias towards focused research over exhaustive exploration"""
    ```
  </Step>

  <Step title="Create the agent">
    Add the model initialization and agent creation to `agent.py`. Choose your provider:

    <Tabs>
      <Tab title="Claude">
        <CodeGroup>
          ```python Google theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
          from datetime import datetime

          from deepagents import create_deep_agent
          from langchain.chat_models import init_chat_model

          max_concurrent_research_units = 3
          max_researcher_iterations = 3

          current_date = datetime.now().strftime("%Y-%m-%d")

          INSTRUCTIONS = (
              RESEARCH_WORKFLOW_INSTRUCTIONS
              + "\n\n"
              + "=" * 80
              + "\n\n"
              + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
                  max_concurrent_research_units=max_concurrent_research_units,
                  max_researcher_iterations=max_researcher_iterations,
              )
          )

          research_sub_agent = {
              "name": "research-agent",
              "description": "Delegate research to the sub-agent. Give one topic at a time.",
              "system_prompt": RESEARCHER_INSTRUCTIONS.format(date=current_date),
              "tools": [tavily_search],
          }

          model = init_chat_model(model="google_genai:gemini-3.5-flash", temperature=0.0)

          agent = create_deep_agent(
              model=model,
              tools=[tavily_search],
              system_prompt=INSTRUCTIONS,
              subagents=[research_sub_agent],
          )
          ```

          ```python OpenAI theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
          from datetime import datetime

          from deepagents import create_deep_agent
          from langchain.chat_models import init_chat_model

          max_concurrent_research_units = 3
          max_researcher_iterations = 3

          current_date = datetime.now().strftime("%Y-%m-%d")

          INSTRUCTIONS = (
              RESEARCH_WORKFLOW_INSTRUCTIONS
              + "\n\n"
              + "=" * 80
              + "\n\n"
              + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
                  max_concurrent_research_units=max_concurrent_research_units,
                  max_researcher_iterations=max_researcher_iterations,
              )
          )

          research_sub_agent = {
              "name": "research-agent",
              "description": "Delegate research to the sub-agent. Give one topic at a time.",
              "system_prompt": RESEARCHER_INSTRUCTIONS.format(date=current_date),
              "tools": [tavily_search],
          }

          model = init_chat_model(model="openai:gpt-5.4", temperature=0.0)

          agent = create_deep_agent(
              model=model,
              tools=[tavily_search],
              system_prompt=INSTRUCTIONS,
              subagents=[research_sub_agent],
          )
          ```

          ```python Anthropic theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
          from datetime import datetime

          from deepagents import create_deep_agent
          from langchain.chat_models import init_chat_model

          max_concurrent_research_units = 3
          max_researcher_iterations = 3

          current_date = datetime.now().strftime("%Y-%m-%d")

          INSTRUCTIONS = (
              RESEARCH_WORKFLOW_INSTRUCTIONS
              + "\n\n"
              + "=" * 80
              + "\n\n"
              + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
                  max_concurrent_research_units=max_concurrent_research_units,
                  max_researcher_iterations=max_researcher_iterations,
              )
          )

          research_sub_agent = {
              "name": "research-agent",
              "description": "Delegate research to the sub-agent. Give one topic at a time.",
              "system_prompt": RESEARCHER_INSTRUCTIONS.format(date=current_date),
              "tools": [tavily_search],
          }

          model = init_chat_model(model="anthropic:claude-sonnet-4-6", temperature=0.0)

          agent = create_deep_agent(
              model=model,
              tools=[tavily_search],
              system_prompt=INSTRUCTIONS,
              subagents=[research_sub_agent],
          )
          ```

          ```python OpenRouter theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
          from datetime import datetime

          from deepagents import create_deep_agent
          from langchain.chat_models import init_chat_model

          max_concurrent_research_units = 3
          max_researcher_iterations = 3

          current_date = datetime.now().strftime("%Y-%m-%d")

          INSTRUCTIONS = (
              RESEARCH_WORKFLOW_INSTRUCTIONS
              + "\n\n"
              + "=" * 80
              + "\n\n"
              + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
                  max_concurrent_research_units=max_concurrent_research_units,
                  max_researcher_iterations=max_researcher_iterations,
              )
          )

          research_sub_agent = {
              "name": "research-agent",
              "description": "Delegate research to the sub-agent. Give one topic at a time.",
              "system_prompt": RESEARCHER_INSTRUCTIONS.format(date=current_date),
              "tools": [tavily_search],
          }

          model = init_chat_model(model="openrouter:anthropic/claude-sonnet-4-6", temperature=0.0)

          agent = create_deep_agent(
              model=model,
              tools=[tavily_search],
              system_prompt=INSTRUCTIONS,
              subagents=[research_sub_agent],
          )
          ```

          ```python Fireworks theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
          from datetime import datetime

          from deepagents import create_deep_agent
          from langchain.chat_models import init_chat_model

          max_concurrent_research_units = 3
          max_researcher_iterations = 3

          current_date = datetime.now().strftime("%Y-%m-%d")

          INSTRUCTIONS = (
              RESEARCH_WORKFLOW_INSTRUCTIONS
              + "\n\n"
              + "=" * 80
              + "\n\n"
              + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
                  max_concurrent_research_units=max_concurrent_research_units,
                  max_researcher_iterations=max_researcher_iterations,
              )
          )

          research_sub_agent = {
              "name": "research-agent",
              "description": "Delegate research to the sub-agent. Give one topic at a time.",
              "system_prompt": RESEARCHER_INSTRUCTIONS.format(date=current_date),
              "tools": [tavily_search],
          }

          model = init_chat_model(model="fireworks:accounts/fireworks/models/qwen3p5-397b-a17b", temperature=0.0)

          agent = create_deep_agent(
              model=model,
              tools=[tavily_search],
              system_prompt=INSTRUCTIONS,
              subagents=[research_sub_agent],
          )
          ```

          ```python Baseten theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
          from datetime import datetime

          from deepagents import create_deep_agent
          from langchain.chat_models import init_chat_model

          max_concurrent_research_units = 3
          max_researcher_iterations = 3

          current_date = datetime.now().strftime("%Y-%m-%d")

          INSTRUCTIONS = (
              RESEARCH_WORKFLOW_INSTRUCTIONS
              + "\n\n"
              + "=" * 80
              + "\n\n"
              + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
                  max_concurrent_research_units=max_concurrent_research_units,
                  max_researcher_iterations=max_researcher_iterations,
              )
          )

          research_sub_agent = {
              "name": "research-agent",
              "description": "Delegate research to the sub-agent. Give one topic at a time.",
              "system_prompt": RESEARCHER_INSTRUCTIONS.format(date=current_date),
              "tools": [tavily_search],
          }

          model = init_chat_model(model="baseten:zai-org/GLM-5.2", temperature=0.0)

          agent = create_deep_agent(
              model=model,
              tools=[tavily_search],
              system_prompt=INSTRUCTIONS,
              subagents=[research_sub_agent],
          )
          ```

          ```python Ollama theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
          from datetime import datetime

          from deepagents import create_deep_agent
          from langchain.chat_models import init_chat_model

          max_concurrent_research_units = 3
          max_researcher_iterations = 3

          current_date = datetime.now().strftime("%Y-%m-%d")

          INSTRUCTIONS = (
              RESEARCH_WORKFLOW_INSTRUCTIONS
              + "\n\n"
              + "=" * 80
              + "\n\n"
              + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
                  max_concurrent_research_units=max_concurrent_research_units,
                  max_researcher_iterations=max_researcher_iterations,
              )
          )

          research_sub_agent = {
              "name": "research-agent",
              "description": "Delegate research to the sub-agent. Give one topic at a time.",
              "system_prompt": RESEARCHER_INSTRUCTIONS.format(date=current_date),
              "tools": [tavily_search],
          }

          model = init_chat_model(model="ollama:devstral-2", temperature=0.0)

          agent = create_deep_agent(
              model=model,
              tools=[tavily_search],
              system_prompt=INSTRUCTIONS,
              subagents=[research_sub_agent],
          )
          ```
        </CodeGroup>
      </Tab>

      <Tab title="Gemini">
        ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
        from datetime import datetime

        from langchain_google_genai import ChatGoogleGenerativeAI
        from deepagents import create_deep_agent

        max_concurrent_research_units = 3
        max_researcher_iterations = 3

        current_date = datetime.now().strftime("%Y-%m-%d")

        INSTRUCTIONS = (
            RESEARCH_WORKFLOW_INSTRUCTIONS
            + "\n\n"
            + "=" * 80
            + "\n\n"
            + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
                max_concurrent_research_units=max_concurrent_research_units,
                max_researcher_iterations=max_researcher_iterations,
            )
        )

        research_sub_agent = {
            "name": "research-agent",
            "description": "Delegate research to the sub-agent. Give one topic at a time.",
            "system_prompt": RESEARCHER_INSTRUCTIONS.format(date=current_date),
            "tools": [tavily_search],
        }

        model = ChatGoogleGenerativeAI(model="gemini-3-pro-preview", temperature=0.0)

        agent = create_deep_agent(
            model=model,
            tools=[tavily_search],
            system_prompt=INSTRUCTIONS,
            subagents=[research_sub_agent],
        )
        ```
      </Tab>
    </Tabs>
  </Step>
</Steps>

## Run the agent

You can run the agent synchronously, meaning it will wait for the full result and then print it, or you can stream updates as they come in.

Add the code from the respective tab at the bottom of `agent.py`:

<Tabs>
  <Tab title="Run synchronously" value="sync">
    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from langchain.messages import HumanMessage

    if __name__ == "__main__":
        result = agent.invoke(
            {
                "messages": [
                    HumanMessage(
                        content="What are the main differences between RAG and fine-tuning for LLM applications?"
                    )
                ]
            }
        )

        for msg in result.get("messages", []):
            if hasattr(msg, "content") and msg.content:
                print(msg.content)
    ```
  </Tab>

  <Tab title="Stream updates" value="stream">
    ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
    from langchain.messages import HumanMessage

    if __name__ == "__main__":
        stream = agent.stream_events(
            {
                "messages": [
                    HumanMessage(content="Compare Python vs JavaScript for web development")
                ]
            },
            version="v3",
        )
        for message in stream.messages:
            for token in message.text:
                print(token, end="", flush=True)
    ```
  </Tab>
</Tabs>

Run the agent from the project root:

```sh theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}
python agent.py
```

If you set the `LANGSMITH_API_KEY` environment variable before running, you can view the agent's traces in [LangSmith](/langsmith/observability) to debug and monitor multi-step behavior.

## Full code

View the complete [Deep Research example](https://github.com/langchain-ai/deepagents/tree/main/examples/deep_research) on GitHub.

## Next steps

Now that you've built the agent, customize it by changing the prompt constants in your agent file to adjust the workflow, delegation strategy, or researcher behavior.
You can also tune the delegation limits to allow for more parallel sub-agents or delegation rounds.

For more information on the concepts in this tutorial, check out the following resources:

* [Subagents](/oss/python/deepagents/subagents): Learn how to configure subagents with different tools and prompts
* [Customization](/oss/python/deepagents/customization): Customize models, tools, system prompts, and planning behavior
* [LangSmith](/langsmith/observability): Trace research runs and debug multi-step behavior
* [Deep Research Course](https://academy.langchain.com/courses/deep-research-with-langgraph): Full course on deep research with LangGraph

***

<div className="source-links">
  <Callout icon="terminal-2">
    [Connect these docs](/use-these-docs) to Claude, VSCode, and more via MCP for real-time answers.
  </Callout>

  <Callout icon="edit">
    [Edit this page on GitHub](https://github.com/langchain-ai/docs/edit/main/src/oss/deepagents/deep-research.mdx) or [file an issue](https://github.com/langchain-ai/docs/issues/new/choose).
  </Callout>
</div>


https://docs.langchain.com/oss/python/deepagents/deep-research

In [ ]:
"""."""
from tavily import TavilyClient
from markdownify import markdownify
import os
from dotenv import load_dotenv
from typing import Literal,Annotated
import httpx
from langchain.tools import InjectedToolArg,tool


## API Key is added so you would have to pay for the websearch your agent is doing.
tv_client = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)


def fetch_url_content(url:str,timeout:float=10.0) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        response = httpx.get(url, headers=headers,timeout=timeout)
        response.raise_for_status()
        return markdownify(response.text)
    except Exception as e:
        return f"Error Occured : {e}"



@tool(parse_docstring=True)
def tavily_search(
    query:str,
    max_results : Annotated[int,InjectedToolArg] =3,
    topic: Annotated[
        Literal["general","news","finance"] , InjectedToolArg
    ] = "general"
) -> str:
    """Search the web for information on a given query.

    Uses Tavily to discover relevant URLs, then fetches and returns full webpage content as markdown.

    Args:
        query: Search query to execute
        max_results: Maximum number of results to return (default: 1)
        topic: Topic filter - 'general', 'news', or 'finance' (default: 'general')

    Returns:
        Formatted search results with full webpage content
    """
    top_search_urls = tv_client.search(
        query,
        max_results=max_results,
        topic=topic
    )

    result_texts = []
    for result in top_search_urls.get("results",[]):
        url = result['url']
        title = result['title']
        content = fetch_url_content(url)
        result_texts.append(f"##{title}\n**URL**:{url}\n\n{content}\n\n---")
    
    result_count = len(result_texts)
    output = f"Found {result_count} result(s) for '{query}':\n\n"
    output+= "\n".join(result_texts)
    return output




In [ ]:
## Test Tool as a standalone function by removing the @tool definition.

# websites_data = tavily_search("Vending machine distributor in India")
# print(websites_data)

In [28]:
from prompts import (SUBAGENT_DELEGATION_INSTRUCTIONS,RESEARCHER_INSTRUCTIONS,RESEARCH_WORKFLOW_INSTRUCTIONS)

In [36]:
from langchain_openrouter import ChatOpenRouter
from langchain.chat_models import init_chat_model
open_router_llm = ChatOpenRouter(
    model = "nvidia/nemotron-3-ultra-550b-a55b:free",
    api_key = os.getenv("OPENROUTER_API_KEY")
)

openrouter_gpt_oss_120B = ChatOpenRouter(
    model = "openai/gpt-oss-120b:free",
    api_key = os.getenv("OPENROUTER_API_KEY")
)

google_3_5_flash_model = init_chat_model(
    model="google_genai:gemini-2.5-flash"
)

In [37]:
from langgraph.graph.message import _format_messages
from langchain_core.messages import HumanMessage
from datetime import datetime
from deepagents import create_deep_agent
from utils import  format_messages


datetime.now().strftime("%Y-%m-%d")
max_concurrent_research_units = 3
max_researcher_iterations = 3




DEEP_AGENT_INSTRUCTIONS = (
    RESEARCH_WORKFLOW_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
            max_concurrent_research_units=max_concurrent_research_units,
            max_researcher_iterations=max_researcher_iterations
    )
    )

research_sub_agent={
    "system_prompt":RESEARCHER_INSTRUCTIONS,
    "tools":[tavily_search],
    "name":"research-agent",
    "description": "Delegate research to the sub-agent. Give one topic at a time."
}

deep_agent = create_deep_agent(
    model= openrouter_gpt_oss_120B,
    system_prompt = DEEP_AGENT_INSTRUCTIONS,
    subagents=[research_sub_agent],
    tools=[tavily_search]
)

result = deep_agent.invoke(
    {
        "messages":[
            HumanMessage("Vending Machine Distributer or Seller in India , latest tech")
        ]
    }
)

format_messages(result["messages"])



╭────────────────────────────────── 🧑 Human ──────────────────────────────────╮
│ Vending Machine Distributer or Seller in India , latest tech                 │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────── 📝 AI ────────────────────────────────────╮
│                                                                              │
│                                                                              │
│ 🔧 Tool Call: write_todos                                                    │
│    Args: {                                                                   │
│   "todos": [                                                                 │
│     {                                                                        │
│       "content": "Create /research_request.md with user's query",            │
│       "status": "in_progress"                                                │
│     },                       

In [38]:
latest_message = result["messages"][-1].content

In [39]:
print(latest_message)

**Latest Vending‑Machine Technology for Distributors / Sellers in India (2023‑2024)**  

| Area | Key Innovations / Trends | Representative Sources |
|------|--------------------------|------------------------|
| **Hardware & Form‑Factor** | • Machines now store **400‑500 items** and support **50+ SKUs** in a compact footprint (≈ 0.8 m³). <br>• Modular designs combine **large‑capacity cabinets** with **plug‑and‑play micro‑units** (≈ 8 sq ft) for offices, hospitals, malls, and transit hubs. <br>• Use of **stainless‑steel enclosures**, **thermal management** for fresh‑food items, and **touch‑free dispensing** (robotic arms, conveyor‑type pick‑up). | LinkedIn – “Smart Vending Machines Revolutionize 24/7 Distribution in India” (Wendor)【2†L3-L9】【2†L30-L38】 |
| **Payment Solutions** | • **Fully cash‑less** operation via **UPI**, QR‑code, NFC, and integrated digital‑wallet gateways (Paytm, PhonePe, Google Pay). <br>• Real‑time transaction settlement and receipt generation through cloud‑based 

Next Steps is to take course on https://academy.langchain.com/courses/deep-research-with-langgraph?_gl=1*1pkm80d*_gcl_au*MjIwNzI0MjgwLjE3Nzk1NDg0NDM.*_ga*MTg4MjEzNDM5NC4xNzc5NTQ4NDQz*_ga_47WX3HKKY2*czE3ODIyOTkxNjMkbzQ5JGcxJHQxNzgyMjk5MTYzJGo2MCRsMCRoMA..